# Anonymization Middleware Demo

This notebook demonstrates the `AnonymizationMiddleware` integrated into a LangChain ReAct agent.

**What it shows:**
1. A user message containing PII (name, email, phone) is **anonymized** before reaching the LLM
2. The LLM sees only fake values and responds using them
3. The LLM response is **deanonymized** — original PII values are restored
4. The anonymization mapping table is printed for inspection

**Key classes used:**
- `AnonymizationMiddleware` — the middleware
- `AnonymizationConfig` — configuration (entity types, Faker locale, fuzzy threshold)
- `PresidioDetectorConfig` — Presidio detector settings (which entity types to detect)

In [1]:
%load_ext autoreload
%autoreload 2

import sys

from dotenv import load_dotenv
from rich import print  # noqa: F401

assert load_dotenv(verbose=True)
sys.path.append(".")

## 1. Build the anonymization middleware

In [2]:
from genai_tk.agents.langchain.middleware.anonymization_middleware import (
    AnonymizationConfig,
    AnonymizationMiddleware,
)
from genai_tk.agents.langchain.middleware.presidio_detector import PresidioDetectorConfig

anon_config = AnonymizationConfig(
    detector=PresidioDetectorConfig(
        analyzed_fields=["PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER", "CREDIT_CARD", "LOCATION"],
        enable_spacy=True,
        spacy_model="en_core_web_sm",
    ),
    faker_seed=42,  # reproducible fake values
    faker_locales=["en_US"],
    fuzzy_deanonymize=True,
    fuzzy_threshold=85,
)

middleware = AnonymizationMiddleware(config=anon_config)
print("Middleware created:", middleware)

2026-04-27 10:43:56.870 | INFO     | genai_tk.utils.spacy_model_mngr:setup_spacy_model:103 - SpaCy model 'en_core_web_sm' is available globally


Middleware created: <genai_tk.agents.langchain.middleware.anonymization_middleware.AnonymizationMiddleware object 
at 0x74982944ce30>

## 2. Low-level demo: anonymize / deanonymize directly

In [3]:
DEMO_TEXT = (
    "Hello, my name is John Smith and you can reach me at john.smith@acme.com "
    "or call +1-555-867-5309. I'm based in Paris."
)

THREAD_ID = "demo-thread-1"

anonymized = middleware._anonymize_text(DEMO_TEXT, THREAD_ID)

print("[bold]Original:[/bold]")
print(DEMO_TEXT)
print()
print("[bold]Anonymized (what the LLM sees):[/bold]")
print(anonymized)

2026-04-27 10:43:58.160 | DEBUG    | genai_tk.agents.langchain.middleware.anonymization_middleware:_anonymize_text:249 - [Anonymization] Anonymized 3 entities for thread 'demo-thread-1'


Original:

Hello, my name is John Smith and you can reach me at john.smith@acme.com or call +1-555-867-5309. I'm based in 
Paris.

Anonymized (what the LLM sees):

Hello, my name is Allison Hill and you can reach me at donaldgarcia@example.net or call +1-555-867-5309. I'm based 
in New Roberttown.

In [4]:
from rich.table import Table

mapping = middleware._mapping.get(THREAD_ID, {})

table = Table(title="PII → Fake Mapping")
table.add_column("Original (PII)", style="red")
table.add_column("Fake (sent to LLM)", style="green")

for original, fake in mapping.items():
    table.add_row(original, fake)

print(table)

                PII → Fake Mapping                
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Original (PII)      ┃ Fake (sent to LLM)       ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ John Smith          │ Allison Hill             │
│ john.smith@acme.com │ donaldgarcia@example.net │
│ Paris               │ New Roberttown           │
└─────────────────────┴──────────────────────────┘

In [5]:
# Simulate an LLM response that uses the fake values
fake_name = None
fake_email = None
for orig, fake in mapping.items():
    if "John Smith" in orig:
        fake_name = fake
    if "john.smith" in orig:
        fake_email = fake

simulated_llm_response = (
    f"I've noted your details. I'll send a confirmation to {fake_email or '<fake-email>'} "
    f"for {fake_name or '<fake-name>'}."
)

print("[bold]Simulated LLM response (contains fake values):[/bold]")
print(simulated_llm_response)

deanonymized = middleware._deanonymize_text(simulated_llm_response, THREAD_ID)

print()
print("[bold]Deanonymized response (original PII restored):[/bold]")
print(deanonymized)

Simulated LLM response (contains fake values):

I've noted your details. I'll send a confirmation to donaldgarcia@example.net for Allison Hill.

Deanonymized response (original PII restored):

I've noted your details. I'll send a confirmation to john.smith@acme.com for John Smith.

## 3. Custom Recognizers

`CustomRecognizerConfig` lets you add domain-specific entity types using regex patterns with optional context words. Context words boost the confidence score when they appear near the match.

In [ ]:
from genai_tk.agents.langchain.middleware.presidio_detector import (
    CustomRecognizerConfig,
    PresidioDetectorConfig,
)

# Recognize internal project codes (PRJ-12345) and employee IDs (EMP-A1234)
custom_config = AnonymizationConfig(
    detector=PresidioDetectorConfig(
        analyzed_fields=["PERSON", "EMAIL_ADDRESS"],
        enable_spacy=False,  # fast, no spaCy needed for regex-based recognizers
        custom_recognizers=[
            CustomRecognizerConfig(
                entity_name="PROJECT_CODE",
                patterns=[r"\bPRJ-\d{5}\b"],
                context=["project", "ticket", "issue"],
                score=0.85,
            ),
            CustomRecognizerConfig(
                entity_name="EMPLOYEE_ID",
                patterns=[r"\bEMP-[A-Z]\d{4}\b"],
                context=["employee", "staff", "hr"],
                score=0.9,
            ),
        ],
    ),
    faker_seed=99,
)

custom_mw = AnonymizationMiddleware(config=custom_config)

CUSTOM_TEXT = (
    "Please assign ticket PRJ-42001 to John Doe (john.doe@corp.com). "
    "His employee ID is EMP-A1234 and he reports to EMP-B5678."
)

CUSTOM_THREAD = "custom-recognizer-demo"
anonymized_custom = custom_mw._anonymize_text(CUSTOM_TEXT, CUSTOM_THREAD)

print("[bold]Original:[/bold]")
print(CUSTOM_TEXT)
print()
print("[bold]Anonymized (custom entities replaced):[/bold]")
print(anonymized_custom)

In [ ]:
from rich.table import Table

custom_mapping = custom_mw._mapping.get(CUSTOM_THREAD, {})

table = Table(title="Custom Entity Mapping")
table.add_column("Original", style="red")
table.add_column("Fake (seen by LLM)", style="green")

for original, fake in custom_mapping.items():
    table.add_row(original, fake)

print(table)

# Round-trip: deanonymize
restored = custom_mw._deanonymize_text(anonymized_custom, CUSTOM_THREAD)
print()
print("[bold]Round-trip (deanonymized):[/bold]")
print(restored)
assert restored == CUSTOM_TEXT, "Round-trip failed!"
print("[green]✓ Round-trip OK[/green]")

## 4. Full agent demo with AnonymizationMiddleware

Connect to a real LLM and run a ReAct agent with the middleware attached.

In [6]:
from langchain.agents import create_agent
from langchain_core.tools import tool

from genai_tk.core.llm_factory import get_llm


@tool
def lookup_contact(name: str) -> str:
    """Look up contact information for a person by name."""
    # Fake directory — will receive anonymized names
    return f"Contact for '{name}': Department=Engineering, Extension=4242"


# Reset mapping for a fresh demo thread
AGENT_THREAD = "agent-demo-thread"
middleware.cleanup(AGENT_THREAD)

# Use default LLM (configured in baseline.yaml)
llm = get_llm()

agent = create_agent(
    model=llm,
    tools=[lookup_contact],
    middleware=[middleware],
    system_prompt="You are a helpful assistant. Use the lookup_contact tool to find contact info.",
)

print("Agent ready")

2026-04-27 10:43:59.270 | DEBUG    | genai_tk.core.llm_factory:get_llm:1128 - get LLM:'openai/gpt-oss-120b@openrouter'


Agent ready

In [7]:
from langchain_core.messages import HumanMessage

from genai_tk.agents.langchain.langchain_agent import _extract_content

USER_MESSAGE = "Hi, I'm Alice Johnson (alice.johnson@company.com, +1-555-234-5678). Can you look up my contact record?"

print("[bold cyan]User message (with real PII):[/bold cyan]")
print(USER_MESSAGE)
print()

result = agent.invoke(
    {"messages": [HumanMessage(content=USER_MESSAGE)]},
    config={"configurable": {"thread_id": AGENT_THREAD}},
)


final_response = _extract_content(result)

print("[bold green]Final response (PII restored by deanonymization):[/bold green]")
print(final_response)

User message (with real PII):

Hi, I'm Alice Johnson (alice.johnson@company.com, +1-555-234-5678). Can you look up my contact record?

2026-04-27 10:44:01.772 | DEBUG    | genai_tk.agents.langchain.middleware.anonymization_middleware:_anonymize_text:249 - [Anonymization] Anonymized 2 entities for thread 'agent-demo-thread'


2026-04-27 10:44:03.845 | DEBUG    | genai_tk.agents.langchain.middleware.anonymization_middleware:_anonymize_text:249 - [Anonymization] Anonymized 2 entities for thread 'agent-demo-thread'
2026-04-27 10:44:05.712 | DEBUG    | genai_tk.agents.langchain.middleware.anonymization_middleware:_anonymize_text:249 - [Anonymization] Anonymized 2 entities for thread 'agent-demo-thread'
2026-04-27 10:44:09.321 | DEBUG    | genai_tk.agents.langchain.middleware.anonymization_middleware:_anonymize_text:249 - [Anonymization] Anonymized 2 entities for thread 'agent-demo-thread'
2026-04-27 10:44:11.017 | DEBUG    | genai_tk.agents.langchain.middleware.anonymization_middleware:_anonymize_text:249 - [Anonymization] Anonymized 2 entities for thread 'agent-demo-thread'
2026-04-27 10:44:12.486 | DEBUG    | genai_tk.agents.langchain.middleware.anonymization_middleware:_anonymize_text:249 - [Anonymization] Anonymized 2 entities for thread 'agent-demo-thread'
2026-04-27 10:44:14.102 | DEBUG    | genai_tk.agen

Final response (PII restored by deanonymization):

Here’s the contact information we have on file for **Jessica Holmes**: - **Department:** Engineering - 
**Extension:** 4242 If you need any additional details (e.g., office location, manager, etc.) or want to update 
your information, just let me know!

In [8]:
# Inspect the mapping for this agent thread
agent_mapping = middleware._mapping.get(AGENT_THREAD, {})

table = Table(title=f"Mapping for thread '{AGENT_THREAD}'")
table.add_column("Original (PII)", style="red")
table.add_column("Fake (seen by LLM)", style="green")

for original, fake in agent_mapping.items():
    table.add_row(original, fake)

print(table)

            Mapping for thread 'agent-demo-thread'             
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Original (PII)               ┃ Fake (seen by LLM)           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Alice Johnson                │ David Guzman                 │
│ alice.johnson@company.com    │ jennifermiles@example.com    │
│ David Guzman                 │ Kevin Pacheco                │
│ jennifermiles@example.com    │ blakeerik@example.com        │
│ Kevin Pacheco                │ Christopher Bernard          │
│ blakeerik@example.com        │ curtis61@example.com         │
│ Christopher Bernard          │ Lindsey Roman                │
│ curtis61@example.com         │ yherrera@example.org         │
│ Lindsey Roman                │ Sandra Montgomery            │
│ yherrera@example.org         │ barbara10@example.net        │
│ Sandra Montgomery            │ Dylan Miller                 │
│ barbara10@example.net        │ michellejames@example.com    │
│ Dylan Miller                 │ Holly Wood                   │
│ michellejames@example.com    │ elizabethmiles@example.net   │
│ Holly Wood                   │ Lisa Jackson                 │
│ elizabethmiles@example.net   │ frankgray@example.net        │
│ Lisa Jackson                 │ Colleen Nguyen               │
│ frankgray@example.net        │ perezantonio@example.com     │
│ Colleen Nguyen               │ Austin Gentry                │
│ perezantonio@example.com     │ jason76@example.net          │
│ Austin Gentry                │ Derek Wright                 │
│ jason76@example.net          │ julie69@example.com          │
│ Derek Wright                 │ Zachary Taylor               │
│ julie69@example.com          │ ddavis@example.org           │
│ Zachary Taylor               │ Christopher Becker           │
│ ddavis@example.org           │ cruzcaitlin@example.com      │
│ Christopher Becker           │ Tricia Valencia              │
│ cruzcaitlin@example.com      │ frazierdanny@example.net     │
│ Tricia Valencia              │ Patricia Peterson            │
│ frazierdanny@example.net     │ ryan70@example.net           │
│ Patricia Peterson            │ Deborah Mason                │
│ ryan70@example.net           │ williamrodriguez@example.net │
│ Deborah Mason                │ Mark Lynch                   │
│ williamrodriguez@example.net │ nathanielmartin@example.net  │
│ Mark Lynch                   │ Linda Burns                  │
│ nathanielmartin@example.net  │ hickmannatasha@example.com   │
│ Linda Burns                  │ David Bradley                │
│ hickmannatasha@example.com   │ millertodd@example.org       │
│ David Bradley                │ Kim Martinez                 │
│ millertodd@example.org       │ karroyo@example.com          │
│ Kim Martinez                 │ Lisa Brandt                  │
│ karroyo@example.com          │ jenniferross@example.net     │
│ Lisa Brandt                  │ Jessica Holmes               │
│ jenniferross@example.net     │ wrightcaleb@example.org      │
│ Jessica Holmes               │ Crystal Robinson             │
│ wrightcaleb@example.org      │ zimmermanbrian@example.org   │
│ zimmermanbrian@example.org   │ heathchad@example.org        │
└──────────────────────────────┴──────────────────────────────┘

## 4. YAML configuration equivalent

To use this middleware in a profile defined in `config/agents/langchain.yaml`:

```yaml
langchain_agents:
  profiles:
    - name: PrivacyAgent
      type: react
      llm: default
      middlewares:
        - class: genai_tk.agents.langchain.middleware.anonymization_middleware:AnonymizationMiddleware
          analyzed_fields:
            - PERSON
            - EMAIL_ADDRESS
            - PHONE_NUMBER
            - CREDIT_CARD
          faker_seed: 42
          fuzzy_deanonymize: true
          fuzzy_threshold: 85
```

Then run with:
```bash
uv run cli agents langchain --profile PrivacyAgent --chat
```